# Jarvis OmniVoice Kaggle proof of concept

This notebook compares three private reference recordings (mixed Tamil-English, Tamil-only, English-only) using the official `k2-fsa/OmniVoice` API. It measures prompt preparation and synthesis latency, tests Tamil, English and code-switching, and writes a privacy-safe metrics report. Audio is played inside this private Kaggle notebook; reference audio and generated samples are never written to the GitHub repository.

Before running: attach your private Kaggle dataset containing `mixed.wav`, `tamil.wav`, and `english.wav` under `/kaggle/input/jarvis-reference-audio/`. Set exact transcripts in the configuration cell. Each reference should be a clean 3–10 second recording.

In [ ]:
from pathlib import Path
import json, os, platform, time, hashlib
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, display

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before running this proof of concept.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
%pip install -q omnivoice soundfile pandas


## Private reference configuration

Replace the three transcript strings with the exact spoken words in each recording. Do not upload recordings you do not own or have permission to use. The notebook refuses to continue while placeholders remain.

In [ ]:
REFERENCE_ROOT = Path('/kaggle/input/jarvis-reference-audio')
REFERENCES = {
    'mixed': {'path': REFERENCE_ROOT / 'mixed.wav', 'text': 'PASTE THE EXACT TAMIL AND ENGLISH TRANSCRIPT HERE'},
    'tamil': {'path': REFERENCE_ROOT / 'tamil.wav', 'text': 'இங்கே தமிழ் பதிவின் சரியான உரையை இடவும்'},
    'english': {'path': REFERENCE_ROOT / 'english.wav', 'text': 'PASTE THE EXACT ENGLISH TRANSCRIPT HERE'},
}
if any('PASTE THE EXACT' in item['text'] or 'இங்கே தமிழ்' in item['text'] for item in REFERENCES.values()):
    raise ValueError('Set the exact transcript for all three reference recordings before running.')
for name, item in REFERENCES.items():
    if not item['path'].is_file():
        raise FileNotFoundError(f'Missing {name} reference: {item["path"]}')
    info = sf.info(item['path'])
    if not 3.0 <= info.duration <= 10.0:
        raise ValueError(f'{name} reference is {info.duration:.2f}s; use a clean 3–10s clip.')
    if info.channels > 2:
        raise ValueError(f'{name} reference must be mono or stereo, not {info.channels} channels.')
    print(name, f'{info.duration:.2f}s', info.samplerate, 'Hz', info.channels, 'channel(s)')


## Load model and prepare reusable voice prompts

OmniVoice returns completed waveform arrays rather than token-level streaming audio. This experiment measures full-phrase synthesis; the Android integration will queue generated phrases and play each as soon as it is ready.

In [ ]:
from omnivoice import OmniVoice

load_started = time.perf_counter()
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map='cuda:0', dtype=torch.float16)
torch.cuda.synchronize()
model_load_seconds = time.perf_counter() - load_started
print(f'Model load: {model_load_seconds:.2f}s')

prompts = {}
prompt_metrics = []
for name, item in REFERENCES.items():
    started = time.perf_counter()
    prompts[name] = model.create_voice_clone_prompt(ref_audio=str(item['path']), ref_text=item['text'])
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    prompts[name].save(f'/kaggle/temp/jarvis_{name}_prompt.pt')
    prompt_metrics.append({'reference': name, 'prompt_seconds': round(elapsed, 3)})
    print(name, f'prompt prepared in {elapsed:.2f}s')


## Tamil, English and code-switch tests

The same utterances are synthesized with each reference prompt. Tamil and English are explicitly labelled; the mixed sentence uses language-agnostic mode to test natural code-switching. Listen to the outputs in this private notebook and compare identity consistency, Tamil pronunciation, English pronunciation and transitions.

In [ ]:
TESTS = [
    {'id': 'tamil', 'text': 'அட, சரி. இப்போ என்ன பண்ணலாம்?', 'language': 'Tamil'},
    {'id': 'english', 'text': "Okay, I'll check that and get back to you.", 'language': 'English'},
    {'id': 'mixed', 'text': "அது சரி, but first let's check the API response.", 'language': None},
    {'id': 'mixed_reaction', 'text': '[confirmation-en] ஆமா, that makes sense.', 'language': None},
]
results = []
for ref_name, prompt in prompts.items():
    for case in TESTS:
        started = time.perf_counter()
        audio = model.generate(text=case['text'], language=case['language'], voice_clone_prompt=prompt, num_step=16, speed=1.0)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        waveform = np.asarray(audio[0], dtype=np.float32)
        audio_seconds = len(waveform) / 24000.0
        rtf = elapsed / max(audio_seconds, 1e-6)
        results.append({'reference': ref_name, 'test': case['id'], 'steps': 16, 'synthesis_seconds': round(elapsed, 3), 'audio_seconds': round(audio_seconds, 3), 'rtf': round(rtf, 3), 'sample_rate': 24000})
        print(f'REFERENCE={ref_name} TEST={case["id"]} LATENCY={elapsed:.2f}s AUDIO={audio_seconds:.2f}s RTF={rtf:.3f}')
        display(Audio(waveform, rate=24000))


## Step-count latency comparison

Compare 16 and 32 diffusion steps for the same mixed-language phrase using each reference. This is a latency/quality listening comparison, not an automatic voice-similarity score.

In [ ]:
step_results = []
comparison_text = "அது சரி, let's test the next step carefully."
for ref_name, prompt in prompts.items():
    for steps in (16, 32):
        started = time.perf_counter()
        audio = model.generate(text=comparison_text, language=None, voice_clone_prompt=prompt, num_step=steps, speed=1.0)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        waveform = np.asarray(audio[0], dtype=np.float32)
        audio_seconds = len(waveform) / 24000.0
        row = {'reference': ref_name, 'test': 'mixed_step_comparison', 'steps': steps, 'synthesis_seconds': round(elapsed, 3), 'audio_seconds': round(audio_seconds, 3), 'rtf': round(elapsed / max(audio_seconds, 1e-6), 3), 'sample_rate': 24000}
        step_results.append(row)
        print(row)
        display(Audio(waveform, rate=24000))


## Export reproducible metrics

The JSON report contains timing and runtime metadata only. It does not contain reference audio, generated audio, voice prompts, or transcripts. Audio samples remain in the private Kaggle notebook output.

In [ ]:
all_results = results + step_results
report = {
    'project': 'jarvis-omnivoice-kaggle-poc',
    'model': 'k2-fsa/OmniVoice',
    'model_load_seconds': round(model_load_seconds, 3),
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0), 'gpu_memory_gib': round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)},
    'references': [{'name': name, 'duration_seconds': round(sf.info(item['path']).duration, 3), 'sample_rate': sf.info(item['path']).samplerate} for name, item in REFERENCES.items()],
    'prompt_metrics': prompt_metrics,
    'synthesis_metrics': all_results,
    'notes': ['Voice identity and pronunciation require human listening review.', 'OmniVoice generate() returns completed phrase audio; this is not token-level streaming.']
}
output_path = Path('/kaggle/working/jarvis_omnivoice_metrics.json')
output_path.write_text(json.dumps(report, ensure_ascii=False, indent=2))
display(pd.DataFrame(all_results))
print('METRICS_REPORT=', output_path)
print('REFERENCE_AUDIO_EXPORTED=false')
print('VOICE_PROMPTS_EXPORTED=false')
